## Idea
1. Store alll the evidence into index and train the retriever by label evidence to get the right evidence for testing set
2. Trsin the transformer model to compare claim and evidence to predict the class

For labelled dataset we compare the claim with the given label evidence; for unlabeled dataset(i.e. testing set) we compare claims with our retrieved evidence to predict the class

## Imports and configuration

In [3]:
# !pip install torch transformers scikit-learn rank-bm25 tqdm numpy

In [4]:
import json
import re
import os
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi

import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

LABEL2ID = {
    "SUPPORTS": 0,
    "REFUTES": 1,
    "NOT_ENOUGH_INFO": 2,
    "DISPUTED": 3
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


## Read files

In [5]:
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

EVIDENCE_PATH = DATA_DIR / "evidence.json"
TRAIN_PATH = DATA_DIR / "train-claims.json"
DEV_PATH = DATA_DIR / "dev-claims.json"
TEST_PATH = DATA_DIR / "test-claims-unlabelled.json"

MODEL_NAME = "distilroberta-base"
MODEL_DIR = OUTPUT_DIR / "verifier_model"

DEV_PRED_PATH = OUTPUT_DIR / "dev_predictions.json"
TEST_PRED_PATH = OUTPUT_DIR / "test_predictions.json"

## Utility functions 

load/save file, get token/items, normalise text, concat evidence

In [6]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def normalise_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.\,\-\%°]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def simple_tokenise(text: str) -> List[str]:
    return normalise_text(text).split()


def get_claim_items(claims_json: Dict) -> List[Tuple[str, Dict]]:
    return list(claims_json.items())


def concatenate_evidence(
    evidence_ids: List[str],
    evidence_corpus: Dict[str, str],
    max_evidence: int = 5
) -> str:
    selected_ids = evidence_ids[:max_evidence]
    evidence_texts = []

    for eid in selected_ids:
        if eid in evidence_corpus:
            evidence_texts.append(evidence_corpus[eid])

    if len(evidence_texts) == 0:
        return "No relevant evidence found."

    return " ".join(evidence_texts)

## Load data

In [7]:
evidence_corpus = load_json(EVIDENCE_PATH)
train_claims = load_json(TRAIN_PATH)
dev_claims = load_json(DEV_PATH)

print("Number of evidence passages:", len(evidence_corpus))
print("Number of train claims:", len(train_claims))
print("Number of dev claims:", len(dev_claims))

Number of evidence passages: 1208827
Number of train claims: 1228
Number of dev claims: 154


# Retriever

In [8]:
class BM25Retriever:
    def __init__(self, evidence_corpus: Dict[str, str]):
        self.evidence_corpus = evidence_corpus
        self.evidence_ids = list(evidence_corpus.keys())
        self.evidence_texts = [evidence_corpus[eid] for eid in self.evidence_ids]

        print("Building BM25 index...")
        tokenised_corpus = [simple_tokenise(text) for text in tqdm(self.evidence_texts)]
        self.bm25 = BM25Okapi(tokenised_corpus)

    def retrieve(self, claim_text: str, top_k: int = 5) -> List[str]:
        query_tokens = simple_tokenise(claim_text)
        scores = self.bm25.get_scores(query_tokens)
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [self.evidence_ids[i] for i in top_indices]

    def evaluate_recall_at_k(self, claims_json: Dict, k: int = 5) -> float:
        total = 0
        hit = 0

        for claim_id, instance in tqdm(get_claim_items(claims_json)):
            claim_text = instance["claim_text"]
            gold_evidence = set(instance.get("evidences", []))

            if len(gold_evidence) == 0:
                continue

            retrieved = set(self.retrieve(claim_text, top_k=k))

            if len(gold_evidence.intersection(retrieved)) > 0:
                hit += 1

            total += 1

        return hit / total if total > 0 else 0.0

##  Build and evaluate retriever

In [9]:
retriever = BM25Retriever(evidence_corpus)

for k in [1, 3, 5, 10]:
    recall = retriever.evaluate_recall_at_k(dev_claims, k=k)
    print(f"Dev Retrieval Recall@{k}: {recall:.4f}")

Building BM25 index...


 60%|█████▉    | 92/154 [18:16<12:18, 11.92s/it]


KeyboardInterrupt: 

# Transformer Verifier

## Get dataset

In [ ]:
class ClaimEvidenceDataset(Dataset):
    def __init__(
        self,
        claims_json: Dict,
        evidence_corpus: Dict[str, str],
        tokenizer,
        max_length: int = 512,
        max_evidence: int = 5,
        use_gold_evidence: bool = True,
        retriever: BM25Retriever = None,
        retrieval_top_k: int = 5,
        is_test: bool = False
    ):
        self.items = get_claim_items(claims_json)
        self.evidence_corpus = evidence_corpus
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_evidence = max_evidence
        self.use_gold_evidence = use_gold_evidence
        self.retriever = retriever
        self.retrieval_top_k = retrieval_top_k
        self.is_test = is_test

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        claim_id, instance = self.items[idx]
        claim_text = instance["claim_text"]

        if self.use_gold_evidence and not self.is_test:
            evidence_ids = instance.get("evidences", [])
        else:
            if self.retriever is None:
                raise ValueError("Retriever is required when not using gold evidence.")
            evidence_ids = self.retriever.retrieve(claim_text, top_k=self.retrieval_top_k)

        evidence_text = concatenate_evidence(
            evidence_ids=evidence_ids,
            evidence_corpus=self.evidence_corpus,
            max_evidence=self.max_evidence
        )

        encoded = self.tokenizer(
            claim_text,
            evidence_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "claim_id": claim_id,
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "evidence_ids": evidence_ids
        }

        if not self.is_test:
            label = instance["claim_label"]
            item["label"] = torch.tensor(LABEL2ID[label], dtype=torch.long)

        return item


def collate_fn(batch):
    input_ids = torch.stack([x["input_ids"] for x in batch])
    attention_mask = torch.stack([x["attention_mask"] for x in batch])
    claim_ids = [x["claim_id"] for x in batch]
    evidence_ids = [x["evidence_ids"] for x in batch]

    output = {
        "claim_ids": claim_ids,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "evidence_ids": evidence_ids
    }

    if "label" in batch[0]:
        labels = torch.stack([x["label"] for x in batch])
        output["labels"] = labels

    return output

## Training and evaluation functions

In [ ]:
def evaluate_verifier(model, dataloader, device: str):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    print(classification_report(
        all_labels,
        all_preds,
        target_names=[ID2LABEL[i] for i in range(4)]
    ))

    return acc, macro_f1


def train_verifier(
    train_claims: Dict,
    dev_claims: Dict,
    evidence_corpus: Dict[str, str],
    model_name: str = "distilroberta-base",
    output_dir: Path = Path("outputs/verifier_model"),
    epochs: int = 3,
    batch_size: int = 8,
    lr: float = 2e-5,
    max_length: int = 512,
    max_evidence: int = 5,
    device: str = "cpu"
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=4,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )
    model.to(device)

    train_dataset = ClaimEvidenceDataset(
        claims_json=train_claims,
        evidence_corpus=evidence_corpus,
        tokenizer=tokenizer,
        max_length=max_length,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False
    )

    dev_dataset = ClaimEvidenceDataset(
        claims_json=dev_claims,
        evidence_corpus=evidence_corpus,
        tokenizer=tokenizer,
        max_length=max_length,
        max_evidence=max_evidence,
        use_gold_evidence=True,
        is_test=False
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    optimiser = torch.optim.AdamW(model.parameters(), lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimiser,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    best_macro_f1 = 0.0

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        model.train()
        total_loss = 0.0

        for batch in tqdm(train_loader):
            optimiser.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimiser.step()
            scheduler.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Training loss: {avg_loss:.4f}")

        dev_acc, dev_macro_f1 = evaluate_verifier(model, dev_loader, device)
        print(f"Dev accuracy with gold evidence: {dev_acc:.4f}")
        print(f"Dev macro F1 with gold evidence: {dev_macro_f1:.4f}")

        if dev_macro_f1 > best_macro_f1:
            best_macro_f1 = dev_macro_f1
            output_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
            print(f"Saved best model to {output_dir}")

    return model, tokenizer

## Train the Transformer verifier

This trains using **gold evidence** from the labelled training data.

In [ ]:
# You can reduce epochs or batch size if training is slow.
model, tokenizer = train_verifier(
    train_claims=train_claims,
    dev_claims=dev_claims,
    evidence_corpus=evidence_corpus,
    model_name=MODEL_NAME,
    output_dir=MODEL_DIR,
    epochs=5,
    batch_size=8,
    lr=2e-5,
    max_length=512,
    max_evidence=5,
    device=device
)

Loading weights: 100%|██████████| 101/101 [00:00<00:00, 6303.66it/s]
RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Epoch 1/3


100%|██████████| 154/154 [21:02<00:00,  8.20s/it]


Training loss: 1.1523


100%|██████████| 20/20 [00:26<00:00,  1.32s/it]
c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

...

Training loss: 0.7669


100%|██████████| 20/20 [00:25<00:00,  1.28s/it]
c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Therese\anaconda3\envs\rag\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

                 precision    recall  f1-score   support

       SUPPORTS       0.57      0.94      0.71        68
        REFUTES       0.00      0.00      0.00        27
NOT_ENOUGH_INFO       0.80      0.80      0.80        41
       DISPUTED       0.00      0.00      0.00        18

       accuracy                           0.63       154
      macro avg       0.34      0.44      0.38       154
   weighted avg       0.46      0.63      0.53       154

Dev accuracy with gold evidence: 0.6299
Dev macro F1 with gold evidence: 0.3780


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.05it/s]

Saved best model to outputs\verifier_model


##  Full pipeline prediction function
This uses retrieved evidence instead of gold evidence. Use this for dev full-pipeline testing and test prediction.

In [ ]:
def predict_claims(
    claims_json: Dict,
    evidence_corpus: Dict[str, str],
    retriever: BM25Retriever,
    model_path: Path,
    output_path: Path,
    retrieval_top_k: int = 10,
    max_evidence: int = 3,
    batch_size: int = 8,
    max_length: int = 512,
    is_test: bool = False,
    device: str = "cpu"
):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(device)
    model.eval()

    dataset = ClaimEvidenceDataset(
        claims_json=claims_json,
        evidence_corpus=evidence_corpus,
        tokenizer=tokenizer,
        max_length=max_length,
        max_evidence=max_evidence,
        use_gold_evidence=False,
        retriever=retriever,
        retrieval_top_k=retrieval_top_k,
        is_test=is_test
    )

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    predictions = {}

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            pred_ids = torch.argmax(outputs.logits, dim=1).cpu().numpy().tolist()

            for claim_id, pred_id, evidence_ids in zip(batch["claim_ids"], pred_ids, batch["evidence_ids"]):
                predictions[claim_id] = {
                    "claim_label": ID2LABEL[pred_id],
                    "evidences": evidence_ids[:max_evidence]
                }

    save_json(predictions, output_path)
    print(f"Saved predictions to {output_path}")
    return predictions

## Generate dev predictions using retrieved evidence
This creates a prediction file that can be evaluated using the provided eval.py script.

note:

retrieval_top_k=10: retrieve 10 candidates internally.

max_evidence=3: only output the best 3 evidence IDs.

This often gives a better precision/recall balance than outputting all 10.

In [ ]:
dev_predictions = predict_claims(
    claims_json=dev_claims,
    evidence_corpus=evidence_corpus,
    retriever=BM25Retriever(evidence_corpus),
    model_path=MODEL_DIR,
    output_path=DEV_PRED_PATH,
    retrieval_top_k=10,
    max_evidence=3,
    batch_size=8,
    max_length=512,
    is_test=False,
    device=device
)

NameError: name 'predict_claims' is not defined

## evaulation by eval.py

In [ ]:
# Run the official evaluator on dev predictions.


!python eval.py --predictions outputs/dev_predictions.json --groundtruth data/dev-claims.json

Evidence Retrieval F-score (F)    = 0.08902288188002473
Claim Classification Accuracy (A) = 0.44155844155844154
Harmonic Mean of F and A          = 0.14817259202130112


## predict test claims

In [ ]:
test_claims = load_json(TEST_PATH)

test_predictions = predict_claims(
    claims_json=test_claims,
    evidence_corpus=evidence_corpus,
    retriever=retriever,
    model_path=MODEL_DIR,
    output_path=TEST_PRED_PATH,
    retrieval_top_k=10,
    max_evidence=3,
    batch_size=8,
    max_length=512,
    is_test=True,
    device=device
)

print("Test prediction file:", TEST_PRED_PATH)

100%|██████████| 20/20 [13:34<00:00, 40.73s/it]

Saved predictions to outputs\test_predictions.json
Test prediction file: outputs\test_predictions.json
